# 04 单独训练一个神经网络（MLP）

这一份做两件事：
1. 训练一个多层感知机（MLP，全连接神经网络），得到"单独 MLP"这个对照组的成绩。
2. 比较两种处理不平衡的方式：什么都不做 vs 把欺诈样本复制多份（随机过采样）。MLP 没有 `class_weight` 参数，所以只能在数据上做文章。

前提：先跑过 02 和 03，Drive 里要有 `train_scaled.csv`、`test_scaled.csv`、`scores_classic.csv`。

## 0. 准备：中文字体 + 读数据

In [ ]:
!apt-get -qq install -y fonts-noto-cjk > /dev/null
import glob
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = glob.glob('/usr/share/fonts/**/NotoSansCJK*-Regular.ttc', recursive=True)[0]
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/data-mining'
for name in ['train_scaled.csv', 'test_scaled.csv', 'scores_classic.csv']:
    assert os.path.exists(f'{DATA_DIR}/{name}'), f'找不到 {name}，请先完整运行 02 和 03'

train_df = pd.read_csv(f'{DATA_DIR}/train_scaled.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test_scaled.csv')
scores = pd.read_csv(f'{DATA_DIR}/scores_classic.csv')
feature_cols = [c for c in train_df.columns if c != 'Class']
X_train, y_train = train_df[feature_cols], train_df['Class']
X_test, y_test = test_df[feature_cols], test_df['Class']
print(f'训练集 {len(train_df)} 行（欺诈 {y_train.sum()}），测试集 {len(test_df)} 行（欺诈 {y_test.sum()}）')

## 1. 网络长什么样

- 输入层：30 个特征。
- 隐藏层：两层，分别 64 个和 32 个神经元，激活函数用 ReLU（负数变 0，正数不变）。最后那层 32 维的输出，就是后面串行组合要拿来喂给 SVM 的"隐空间"。
- 输出层：1 个数，欺诈概率。
- 训练：用 Adam 优化器，把全部训练数据过 20 遍（20 个 epoch）。

为什么不开 sklearn 自带的"早停"：它是看验证集的**准确率**来决定什么时候停，而在这份数据上准确率永远是 99.8% 左右，根本看不出好坏。所以改成固定训练 20 遍，再看损失曲线有没有降下来。

In [ ]:
from sklearn.neural_network import MLPClassifier

def make_mlp():
    return MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                         max_iter=20, random_state=42)

## 2. 方案 A：不做任何不平衡处理

大约要跑 2~4 分钟。训练时会提示"没有收敛"（ConvergenceWarning），这是因为我们故意只训练 20 遍，可以忽略。

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

start = time.time()
mlp_a = make_mlp().fit(X_train, y_train)
time_a = time.time() - start
score_a = mlp_a.predict_proba(X_test)[:, 1]
print(f'训练耗时 {time_a:.1f} 秒')

## 3. 方案 B：把欺诈样本复制 20 份（随机过采样）

训练集里只有 347 条欺诈，复制成约 6900 条，欺诈占比从 0.17% 升到约 3.4%。只对训练集做，测试集不动。
风险：同一条欺诈被模型看了 20 遍，容易被"背下来"，也就是过拟合。看测试集成绩就知道有没有发生。

In [ ]:
fraud = train_df[train_df.Class == 1]
train_b = pd.concat([train_df[train_df.Class == 0]] + [fraud] * 20).sample(frac=1, random_state=42)
print(f'过采样后训练集 {len(train_b)} 行，欺诈占比 {train_b.Class.mean():.2%}')

start = time.time()
mlp_b = make_mlp().fit(train_b[feature_cols], train_b['Class'])
time_b = time.time() - start
score_b = mlp_b.predict_proba(X_test)[:, 1]
print(f'训练耗时 {time_b:.1f} 秒')

## 4. 损失曲线：训练有没有在进步

**横轴**：训练到第几遍（epoch）。**纵轴**：训练损失，越低说明模型在训练集上错得越少。
看什么：曲线是不是一直往下走、最后趋于平缓。如果后面还在明显下降，说明 20 遍可能不够。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(mlp_a.loss_curve_) + 1), mlp_a.loss_curve_, marker='o', label='方案 A：不处理')
ax.plot(range(1, len(mlp_b.loss_curve_) + 1), mlp_b.loss_curve_, marker='o', label='方案 B：欺诈复制 20 份')
ax.set_xlabel('训练遍数（epoch）')
ax.set_ylabel('训练损失')
ax.set_title('MLP 训练损失曲线')
ax.legend()
plt.tight_layout()
plt.show()

## 5. 成绩：默认阈值 0.5 下的笔数 + AUPRC

和 03 一样，把比例换成具体笔数。最后两行是 02 的随机森林和 SVM，拿来对照。

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

def count_at(score, threshold):
    pred = score >= threshold
    fraud = y_test.values == 1
    caught = int((pred & fraud).sum())
    false_alarm = int((pred & ~fraud).sum())
    flagged = caught + false_alarm
    return {
        '抓到': caught,
        '漏掉': int((~pred & fraud).sum()),
        '误报': false_alarm,
        'Recall': caught / fraud.sum(),
        'Precision': caught / flagged if flagged else float('nan'),
        'AUPRC': average_precision_score(y_test, score),
    }

rows = {
    'MLP 方案 A：不处理': count_at(score_a, 0.5),
    'MLP 方案 B：欺诈复制 20 份': count_at(score_b, 0.5),
    '随机森林（02）': count_at(scores['随机森林'].values, 0.5),
    'SVM（02）': count_at(scores['SVM'].values, 0.0),
}
pd.DataFrame(rows).T.round(4)

## 6. PR 曲线对比

**横轴**：Recall（欺诈抓到的比例）。**纵轴**：Precision（喊欺诈的里面真是欺诈的比例）。越靠右上角越好。
看什么：MLP 和随机森林、SVM 比谁的曲线更靠外；A、B 两种做法的曲线差别大不大。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
curves = [
    ('MLP 方案 A', score_a, 'tab:purple'),
    ('MLP 方案 B', score_b, 'tab:pink'),
    ('随机森林', scores['随机森林'].values, 'tab:green'),
    ('SVM', scores['SVM'].values, 'tab:orange'),
]
for name, s, color in curves:
    p, r, _ = precision_recall_curve(y_test, s)
    ax.plot(r, p, color=color, label=f'{name}（AUPRC={average_precision_score(y_test, s):.3f}）')
ax.set_xlabel('Recall（欺诈抓到的比例）')
ax.set_ylabel('Precision（喊欺诈的里面真是欺诈的比例）')
ax.set_title('MLP 与基线模型的 PR 曲线')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 7. 保存，给后面的组合实验用

把两个 MLP 的分数和训练好的模型都存到 Drive。串行组合要用模型提取隐藏层，并行组合要用分数。

In [ ]:
import joblib

scores['MLP_A'] = score_a
scores['MLP_B'] = score_b
scores.to_csv(f'{DATA_DIR}/scores_classic.csv', index=False)
joblib.dump(mlp_a, f'{DATA_DIR}/mlp_a.joblib')
joblib.dump(mlp_b, f'{DATA_DIR}/mlp_b.joblib')
print('已保存：分数追加到 scores_classic.csv，模型存为 mlp_a.joblib / mlp_b.joblib')

## 看完之后记下来（用你自己的话）

1. 损失曲线有没有降下来？20 遍够不够？
2. 不处理的 MLP，默认阈值下是"保守"还是"激进"？和随机森林像不像？
3. 欺诈复制 20 份之后，抓到和误报怎么变了？AUPRC 变了多少？这和 03 第 5 节"加权重不完全等于调阈值"的结论能对上吗？
4. MLP 的 AUPRC 和随机森林、SVM 比，排第几？
5. 报告思考题 2 问"SVM 和神经网络各自的优势"，从这里的结果能看出什么？